# Refine dataset bằng OpenRouter VLM

Notebook này refine `FinalDataset/claims_merged.csv` bằng 2 model qua OpenRouter/OpenAI SDK:

- `google/gemini-2.5-flash`
- `openai/gpt-4o-mini`

Input chính chỉ dùng `claim` và `image`. Các cột khác trong dataset gốc được bỏ qua để tránh leak nhãn/evidence khi refine.

## 1. Cài dependency

In [60]:
# Chạy cell này một lần nếu môi trường chưa có dependency.
# %pip install -U openai pandas pillow tqdm python-dotenv pydantic


## 2. Config

In [61]:
from pathlib import Path
import os

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "refined":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

# Dataset root contains: media/, news_media/, claims_merged.csv
DATASET_ROOT = Path("../FinalDataset")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path("FinalDataset")

INPUT_CSV = DATASET_ROOT / "claims_merged.csv"
ID_COLUMN = "id"
TEXT_COLUMN = "claim"
IMAGE_COLUMN = "image"

# Keep this True for a cheap smoke run before processing the full dataset.
SMOKE_TEST = False
SMOKE_ROWS = 3
LIMIT_ROWS = SMOKE_ROWS if SMOKE_TEST else None

OUTPUT_DIR = Path("refined_outputs_openrouter_smoke" if SMOKE_TEST else "refined_outputs_openrouter")
SAVE_EVERY = 1 if SMOKE_TEST else 25
RETRY_LIMIT = 3
MAX_CONCURRENT_REQUESTS = 6 if SMOKE_TEST else 6

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")

# Both selected models support text+image input and text output on OpenRouter.
# Fast multimodal defaults for refine:
# - Gemini 2.5 Flash: strong fast baseline
# - GPT-4o mini: fast and reliable structured output
MODEL_CONFIGS = [
    {"alias": "gemini-2.5-flash", "model_id": "google/gemini-2.5-flash"},
    {"alias": "gpt4o_mini", "model_id": "openai/gpt-4o-mini"},
]

# Edit this list if you only want to run a subset, for example ["gemini-2.5-flash"].
RUN_MODEL_ALIASES = ["gemini-2.5-flash", "gpt4o_mini"]

TEMPERATURE = 0.1
MAX_TOKENS = 2200 if SMOKE_TEST else 3200

# Resize images before base64 encoding to reduce request size and cost.
MAX_IMAGE_SIDE = 768
JPEG_QUALITY = 70

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Import và OpenRouter client

In [62]:
import ast
import asyncio
import base64
import io
import json
import mimetypes
import re
from typing import Any, Literal

import pandas as pd
from openai import AsyncOpenAI
from PIL import Image
from pydantic import BaseModel, ConfigDict, ValidationError
from tqdm.auto import tqdm

if not OPENROUTER_API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY before running API cells.")

client = AsyncOpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)


## 4. Schema structured output

In [63]:
class ClaimAtom(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str
    text: str
    check_type: Literal["entity", "event", "time", "location", "number", "quote", "relation", "other"]
    priority: Literal["high", "medium", "low"]
    retrieval_queries: list[str]


class VisualObservation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str
    text: str
    visible_evidence: list[str]
    confidence: Literal["high", "medium", "low"]


class Alignment(BaseModel):
    model_config = ConfigDict(extra="forbid")

    label: Literal["match", "partial_match", "mismatch", "not_enough_visual_info"]
    text: str


class KeyEntities(BaseModel):
    model_config = ConfigDict(extra="forbid")

    people: list[str]
    organizations: list[str]
    locations: list[str]
    dates: list[str]
    numbers: list[str]
    other: list[str]


class SearchQueries(BaseModel):
    model_config = ConfigDict(extra="forbid")

    semantic: list[str]
    keywords: list[str]
    visual: list[str]


class RetrievalFocus(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: bool
    image: bool
    cross_modal: bool


class Constraints(BaseModel):
    model_config = ConfigDict(extra="forbid")

    time: list[str]
    location: list[str]
    source_type: list[str]


class RefineOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    original_claim: str
    normalized_claim: str
    primary_retrieval_query: str
    image_provided: bool
    language: Literal["vi"]
    claim_atoms: list[ClaimAtom]
    visual_observations: list[VisualObservation]
    alignment: Alignment
    key_entities: KeyEntities
    search_queries: SearchQueries
    retrieval_focus: RetrievalFocus
    constraints: Constraints
    context_summary: str
    ambiguity_notes: list[str]
    verification_targets: list[str]


REFINE_JSON_SCHEMA = RefineOutput.model_json_schema()


## 5. Prompt tiếng Anh

In [64]:
SYSTEM_INSTRUCTION = """You are a fact-checking input refiner for a Vietnamese multimodal RAG pipeline.
Return only valid JSON that follows the requested schema.
Do not decide the final truth label of the claim.
Do not use external knowledge. Use only the provided claim text and visible image content.
Write all generated field values in accented Vietnamese."""


def build_refine_prompt(claim: str, image_count: int) -> str:
    image_provided = image_count > 0
    return f"""
# Task

Refine a noisy Vietnamese fact-checking input into a clean, structured representation for downstream RAG testing.

# Input

- Claim: {claim}
- Image provided: {str(image_provided).lower()}
- Number of provided images: {image_count}

# Rules

- Write all generated field values in accented Vietnamese.
- The `language` field must be exactly "vi".
- Keep the JSON compact: no markdown, no commentary, no repeated whitespace, no long paragraphs.
- Preserve named entities, numbers, dates, quoted text, locations, and distinctive visual details.
- Normalize the claim without changing its meaning.
- Split the claim into atomic, verifiable facts.
- Create one best primary retrieval query for the whole input.
- Create at most 5 claim_atoms.
- Create at most 2 retrieval query variants for each atomic claim.
- If one or more images are provided, describe only visible evidence across all provided images.
- If multiple images are provided, include observations from all relevant images and keep each observation grounded in visible details.
- Create at most 3 visual_observations and at most 4 visible_evidence items per observation.
- Do not infer identity, intent, unseen events, or off-image context from images.
- Explain whether visible image evidence appears to support, partially support, contradict, or not sufficiently address the claim.
- If no image is provided, use an empty visual_observations array and set alignment.label to "not_enough_visual_info".
- Do not decide the final truth of the claim.
- Create general search query variants for the next phase. Do not mention any vector database, embedding model, or retrieval backend.
- Create at most 3 semantic queries, 6 keyword queries, and 3 visual queries.
- Set retrieval_focus booleans based on which modalities are useful for retrieval.
- Extract explicit time, location, and source-type constraints only when they appear in the input or are clearly required by the claim.
- Use empty arrays when information is absent.
- Keep context_summary to 1-2 short Vietnamese sentences.
- Create at most 5 ambiguity_notes and at most 5 verification_targets.

# Output

Return exactly one JSON object that follows the provided JSON schema.
""".strip()


## 6. Đọc CSV và xử lý đường dẫn ảnh

In [65]:
def parse_image_values(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple)):
                return [str(item).strip() for item in parsed if str(item).strip()]
        except (SyntaxError, ValueError):
            pass
    if " | " in text:
        return [part.strip() for part in text.split(" | ") if part.strip()]
    return [text]


def resolve_image_paths(value: Any, dataset_root: Path) -> list[Path]:
    paths = []
    for item in parse_image_values(value):
        path = Path(item)
        if not path.is_absolute():
            path = dataset_root / path
        if path.exists():
            paths.append(path)
    return paths


def image_to_data_url(path: Path) -> str:
    mime = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    if mime not in {"image/jpeg", "image/png", "image/webp"}:
        mime = "image/jpeg"

    with Image.open(path) as img:
        img = img.convert("RGB")
        img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


def preflight_check() -> pd.DataFrame:
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)
    missing = [col for col in [ID_COLUMN, TEXT_COLUMN, IMAGE_COLUMN] if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available columns: {list(df.columns)}")

    sample = df.head(LIMIT_ROWS or 5).copy()
    sample["_parsed_images"] = sample[IMAGE_COLUMN].apply(parse_image_values)
    sample["_resolved_images"] = sample[IMAGE_COLUMN].apply(lambda value: [str(p) for p in resolve_image_paths(value, DATASET_ROOT)])
    sample["_image_count"] = sample["_resolved_images"].apply(len)
    print(f"Rows in CSV: {len(df)}")
    print(f"Rows selected: {len(df.head(LIMIT_ROWS)) if LIMIT_ROWS else len(df)}")
    print(f"Dataset root: {DATASET_ROOT.resolve()}")
    return sample[[ID_COLUMN, TEXT_COLUMN, IMAGE_COLUMN, "_parsed_images", "_resolved_images", "_image_count"]]


preflight_check()


Rows in CSV: 1293
Rows selected: 1293
Dataset root: D:\FactCheckPipeline\FinalDataset


,id,claim,image,_parsed_images,_resolved_images,_image_count
0,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,media/post_1_cmt_img_0.jpg,[media/post_1_cmt_img_0.jpg],[..\FinalDataset\media\post_1_cmt_img_0.jpg],1
1,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,media/post_1_cmt_img_0.jpg,[media/post_1_cmt_img_0.jpg],[..\FinalDataset\media\post_1_cmt_img_0.jpg],1
2,3,Thượng úy Nguyễn Đức Phước là Điều tra viên th...,media/post_1_cmt_img_0.jpg,[media/post_1_cmt_img_0.jpg],[..\FinalDataset\media\post_1_cmt_img_0.jpg],1
3,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,media/post_1_cmt_img_0.jpg,[media/post_1_cmt_img_0.jpg],[..\FinalDataset\media\post_1_cmt_img_0.jpg],1
4,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,media/post_1_cmt_img_0.jpg,[media/post_1_cmt_img_0.jpg],[..\FinalDataset\media\post_1_cmt_img_0.jpg],1


## 7. Parse JSON, validate, retry

In [66]:
def extract_json(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"No JSON object found in model output: {text[:500]}")
        return json.loads(match.group(0))


def validate_refine_json(data: dict) -> dict:
    if data.get("language") in {"Vietnamese", "Tiếng Việt", "vietnamese", "VI"}:
        data["language"] = "vi"
    return RefineOutput.model_validate(data).model_dump()


def flatten_result(result: dict) -> dict:
    flat = {}
    for key, value in result.items():
        output_key = f"refined_{key}"
        if isinstance(value, (dict, list)):
            flat[output_key] = json.dumps(value, ensure_ascii=False)
        else:
            flat[output_key] = value
    return flat


## 8. Gọi OpenRouter bằng OpenAI SDK

In [67]:
def build_messages(claim: str, image_paths: list[Path], retry_note: str = "") -> list[dict]:
    prompt = build_refine_prompt(claim=claim, image_count=len(image_paths))
    if retry_note:
        prompt += f"\n\n# Retry instruction\nThe previous output failed validation: {retry_note}. Return corrected JSON only."

    content = [{"type": "text", "text": prompt}]
    for path in image_paths:
        content.append({"type": "image_url", "image_url": {"url": image_to_data_url(path)}})

    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": content},
    ]


def response_to_text(response: Any) -> str:
    return response.choices[0].message.content or ""


async def call_openrouter(model_id: str, claim: str, image_paths: list[Path], retry_note: str = "") -> str:
    response = await client.chat.completions.create(
        model=model_id,
        messages=build_messages(claim, image_paths, retry_note=retry_note),
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        stream=False,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "fact_check_refine",
                "strict": True,
                "schema": REFINE_JSON_SCHEMA,
            },
        },
    )
    return response_to_text(response)


async def generate_refine_json(model_id: str, claim: str, image_paths: list[Path], retry_limit: int = RETRY_LIMIT) -> tuple[dict, str, str, int]:
    retry_note = ""
    last_raw = ""
    last_error = ""

    for attempt in range(1, retry_limit + 1):
        try:
            raw = await call_openrouter(model_id, claim, image_paths, retry_note=retry_note)
            last_raw = raw
            data = extract_json(raw)
            validated = validate_refine_json(data)
            return validated, raw, "", attempt
        except Exception as exc:
            last_error = str(exc)
            retry_note = last_error[:1200]
            await asyncio.sleep(min(2 * attempt, 8))

    fallback = {
        "original_claim": claim,
        "normalized_claim": claim,
        "primary_retrieval_query": claim,
        "image_provided": bool(image_paths),
        "language": "vi",
        "claim_atoms": [],
        "visual_observations": [],
        "alignment": {"label": "not_enough_visual_info", "text": "Không tạo được JSON hợp lệ sau khi retry."},
        "key_entities": {"people": [], "organizations": [], "locations": [], "dates": [], "numbers": [], "other": []},
        "search_queries": {"semantic": [claim], "keywords": [], "visual": []},
        "retrieval_focus": {"text": True, "image": bool(image_paths), "cross_modal": bool(image_paths)},
        "constraints": {"time": [], "location": [], "source_type": []},
        "context_summary": "Không tạo được refine output hợp lệ.",
        "ambiguity_notes": [last_error],
        "verification_targets": [claim],
    }
    return fallback, last_raw, f"failed validation after {retry_limit} attempts: {last_error}", retry_limit


## 9. Chạy refine cho 2 model

In [68]:
BASE_COLUMNS = [ID_COLUMN, TEXT_COLUMN, IMAGE_COLUMN]
REFINED_COLUMNS = [
    "model_alias",
    "model_name",
    "refined_original_claim",
    "refined_normalized_claim",
    "refined_primary_retrieval_query",
    "refined_image_provided",
    "refined_language",
    "refined_claim_atoms",
    "refined_visual_observations",
    "refined_alignment",
    "refined_key_entities",
    "refined_search_queries",
    "refined_retrieval_focus",
    "refined_constraints",
    "refined_context_summary",
    "refined_ambiguity_notes",
    "refined_verification_targets",
    "refined_input_image_count",
    "refined_resolved_image_paths",
    "validation_attempts",
    "raw_output",
    "refine_error",
]
OUTPUT_COLUMNS = BASE_COLUMNS + REFINED_COLUMNS


def output_path_for(alias: str) -> Path:
    return OUTPUT_DIR / f"refined_{alias}.csv"


def successful_rows(frame: pd.DataFrame, model_id: str) -> pd.DataFrame:
    if "model_name" not in frame.columns:
        return frame.iloc[0:0].copy()
    if "refine_error" not in frame.columns:
        return frame.loc[frame["model_name"] == model_id].copy()
    error_text = frame["refine_error"].fillna("").astype(str).str.strip()
    return frame.loc[(frame["model_name"] == model_id) & (error_text == "")].copy()


def load_completed_ids(output_csv: Path, model_id: str) -> set:
    if not output_csv.exists():
        return set()
    existing = pd.read_csv(output_csv)
    if ID_COLUMN not in existing.columns:
        return set()
    return set(successful_rows(existing, model_id)[ID_COLUMN].astype(str))


def write_rows(rows: list[dict], output_csv: Path) -> None:
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(rows)
    for col in OUTPUT_COLUMNS:
        if col not in frame.columns:
            frame[col] = ""
    frame[OUTPUT_COLUMNS].to_csv(output_csv, index=False, encoding="utf-8-sig")


async def refine_one_row(row: pd.Series, alias: str, model_id: str, semaphore: asyncio.Semaphore) -> dict:
    async with semaphore:
        claim = str(row[TEXT_COLUMN]).strip()
        image_paths = resolve_image_paths(row.get(IMAGE_COLUMN, ""), DATASET_ROOT)
        result, raw, error, attempts = await generate_refine_json(model_id, claim, image_paths)
        return {
            ID_COLUMN: row[ID_COLUMN],
            TEXT_COLUMN: row[TEXT_COLUMN],
            IMAGE_COLUMN: row[IMAGE_COLUMN],
            "model_alias": alias,
            "model_name": model_id,
            **flatten_result(result),
            "refined_input_image_count": len(image_paths),
            "refined_resolved_image_paths": json.dumps([str(path) for path in image_paths], ensure_ascii=False),
            "validation_attempts": attempts,
            "raw_output": raw,
            "refine_error": error,
        }


async def run_one_model(config: dict) -> pd.DataFrame:
    alias = config["alias"]
    model_id = config["model_id"]
    output_csv = output_path_for(alias)

    df = pd.read_csv(INPUT_CSV)
    if LIMIT_ROWS:
        df = df.head(LIMIT_ROWS).copy()

    rows = []
    if output_csv.exists():
        rows = successful_rows(pd.read_csv(output_csv), model_id).to_dict("records")
    completed_ids = load_completed_ids(output_csv, model_id)

    todo = df[~df[ID_COLUMN].astype(str).isin(completed_ids)].copy()
    print(f"{alias}: {len(completed_ids)} completed, {len(todo)} remaining, output={output_csv}")

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    tasks = [
        asyncio.create_task(refine_one_row(row, alias, model_id, semaphore))
        for _, row in todo.iterrows()
    ]

    completed_since_save = 0
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=alias):
        rows.append(await task)
        completed_since_save += 1
        if completed_since_save >= SAVE_EVERY:
            write_rows(rows, output_csv)
            completed_since_save = 0

    write_rows(rows, output_csv)
    return pd.DataFrame(rows)


selected_configs = [cfg for cfg in MODEL_CONFIGS if cfg["alias"] in RUN_MODEL_ALIASES]
if not selected_configs:
    raise ValueError("RUN_MODEL_ALIASES did not match any MODEL_CONFIGS.")

all_outputs = []
for config in selected_configs:
    all_outputs.append(await run_one_model(config))

combined = pd.concat(all_outputs, ignore_index=True) if all_outputs else pd.DataFrame(columns=OUTPUT_COLUMNS)
combined_path = OUTPUT_DIR / "refined_openrouter_combined.csv"
write_rows(combined.to_dict("records"), combined_path)
print(f"Combined output: {combined_path}")
combined.head()


gemini-2.5-flash: 1293 completed, 0 remaining, output=refined_outputs_openrouter\refined_gemini-2.5-flash.csv


gemini-2.5-flash: 0it [00:00, ?it/s]

gpt4o_mini: 1288 completed, 5 remaining, output=refined_outputs_openrouter\refined_gpt4o_mini.csv


gpt4o_mini:   0%|          | 0/5 [00:00<?, ?it/s]

Combined output: refined_outputs_openrouter\refined_openrouter_combined.csv


,id,claim,image,model_alias,model_name,refined_original_claim,refined_normalized_claim,refined_primary_retrieval_query,refined_image_provided,refined_language,...,refined_retrieval_focus,refined_constraints,refined_context_summary,refined_ambiguity_notes,refined_verification_targets,refined_input_image_count,refined_resolved_image_paths,validation_attempts,raw_output,refine_error
0,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,google/gemini-2.5-flash,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,True,vi,...,"{""text"": true, ""image"": true, ""cross_modal"": t...","{""time"": [], ""location"": [], ""source_type"": []}",Yêu cầu kiểm tra thông tin về tổng số tiền bị ...,[],"[""số tiền bị chiếm đoạt"", ""đường dây lừa đảo""]",1,"[""..\\FinalDataset\\media\\post_1_cmt_img_0.jpg""]",1,"{""original_claim"": ""Tổng số tiền bị chiếm đoạt...",NaN
1,6,Đường dây lừa đảo này chỉ nhắm mục tiêu vào ng...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,google/gemini-2.5-flash,Đường dây lừa đảo này chỉ nhắm mục tiêu vào ng...,Đường dây lừa đảo này chỉ nhắm mục tiêu vào ng...,Đường dây lừa đảo chỉ nhắm mục tiêu người bị h...,True,vi,...,"{""text"": true, ""image"": false, ""cross_modal"": ...","{""time"": [], ""location"": [""Thái Bình""], ""sourc...",Yêu cầu kiểm tra một tuyên bố về việc một đườn...,[],"[""phạm vi hoạt động của đường dây lừa đảo"", ""đ...",1,"[""..\\FinalDataset\\media\\post_1_cmt_img_0.jpg""]",1,"{""original_claim"": ""Đường dây lừa đảo này chỉ ...",NaN
2,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,google/gemini-2.5-flash,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,Các đối tượng lừa đảo bị bắt giữ ngày 15 tháng...,True,vi,...,"{""text"": true, ""image"": true, ""cross_modal"": t...","{""time"": [""15 tháng 5 năm 2024""], ""location"": ...",Tuyên bố nói rằng các đối tượng lừa đảo đã bị ...,"[""Không rõ danh tính cụ thể của các đối tượng ...","[""Xác nhận việc bắt giữ các đối tượng lừa đảo....",1,"[""..\\FinalDataset\\media\\post_1_cmt_img_0.jpg""]",1,"{""original_claim"": ""Các đối tượng lừa đảo đã b...",NaN
3,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,google/gemini-2.5-flash,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,Công an tỉnh Thái Bình khởi tố 10 đối tượng lừ...,True,vi,...,"{""text"": true, ""image"": true, ""cross_modal"": t...","{""time"": [], ""location"": [""Thái Bình""], ""sourc...",Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,[],"[""việc Công an tỉnh Thái Bình khởi tố 10 đối t...",1,"[""..\\FinalDataset\\media\\post_1_cmt_img_0.jpg""]",1,"{\n ""original_claim"": ""Công an tỉnh Thái Bình...",NaN
4,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,google/gemini-2.5-flash,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,đối tượng cầm đầu đường dây lừa đảo thuê nhà H...,True,vi,...,"{""text"": true, ""image"": false, ""cross_modal"": ...","{""time"": [], ""location"": [""Hà Nội"", ""TP. Hồ Ch...",Yêu cầu xác minh thông tin về việc các đối tượ...,[],"[""việc thuê nhà tại Hà Nội của đường dây lừa đ...",1,"[""..\\FinalDataset\\media\\post_1_cmt_img_0.jpg""]",1,"{""original_claim"": ""Các đối tượng cầm đầu đườn...",NaN


## 10. Kiểm tra nhanh output

In [69]:
combined_path = OUTPUT_DIR / "refined_openrouter_combined.csv"
out = pd.read_csv(combined_path)
print(out.shape)
print(out["model_alias"].value_counts(dropna=False))
display(out[[ID_COLUMN, TEXT_COLUMN, IMAGE_COLUMN, "model_alias", "refined_primary_retrieval_query", "refine_error"]].head(10))


(2586, 25)
model_alias
gemini-2.5-flash    1293
gpt4o_mini          1293
Name: count, dtype: int64


,id,claim,image,model_alias,refined_primary_retrieval_query,refine_error
0,4,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,Tổng số tiền bị chiếm đoạt bởi đường dây lừa đ...,NaN
1,6,Đường dây lừa đảo này chỉ nhắm mục tiêu vào ng...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,Đường dây lừa đảo chỉ nhắm mục tiêu người bị h...,NaN
2,5,Các đối tượng lừa đảo đã bị bắt giữ vào ngày 1...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,Các đối tượng lừa đảo bị bắt giữ ngày 15 tháng...,NaN
3,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,Công an tỉnh Thái Bình khởi tố 10 đối tượng lừ...,NaN
4,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,đối tượng cầm đầu đường dây lừa đảo thuê nhà H...,NaN
5,3,Thượng úy Nguyễn Đức Phước là Điều tra viên th...,media/post_1_cmt_img_0.jpg,gemini-2.5-flash,Thông tin về Thượng úy Nguyễn Đức Phước và vai...,NaN
6,9,"Chỉ có 5 đối tượng, bao gồm Bùi Quốc Ý, bị Phò...",media/post_2_img_0.jpg,gemini-2.5-flash,Phòng Cảnh sát hình sự Công an tỉnh Thanh Hóa ...,NaN
7,8,Băng nhóm tội phạm do Bùi Quốc Ý điều hành đã ...,media/post_2_img_0.jpg,gemini-2.5-flash,Băng nhóm tội phạm Bùi Quốc Ý núp bóng doanh n...,NaN
8,10,Bùi Quốc Ý và tất cả các thành viên băng nhóm ...,media/post_2_img_0.jpg,gemini-2.5-flash,Bùi Quốc Ý và băng nhóm sinh sống hoạt động tạ...,NaN
9,7,"Bùi Quốc Ý, tức Ý Ẻng, là đối tượng cầm đầu bă...",media/post_2_img_0.jpg,gemini-2.5-flash,Bùi Quốc Ý Ý Ẻng cầm đầu băng nhóm tội phạm ng...,NaN
